# Notebook consolidado de la tesis (legado, congelado)

**Desarrollo Fintech e inclusión financiera como predictores del crecimiento económico
regional en Colombia (2017–2021)**

Reemplaza a dos notebooks anteriores del mismo trabajo. Se conserva tal como quedó en agosto de 2026; la versión mantenida vive en `src/iif/legacy/`.

Qué hace distinto:

1. **Los rezagos se construyen UNA sola vez, sobre el panel completo (462 obs).**
   En el notebook de correcciones se recalculaban sobre una muestra ya recortada, lo
   que descartaba 33 observaciones válidas (396 → 363 en el Bloque A).
2. **Ordena por `Fecha`, no por `anio`**, antes de calcular diferencias y rezagos.
3. **Estima el modelo A4** que faltaba en la Tabla 6.
4. **Diagnostica la estructura temporal del PIB** (¿valor anual repetido?).
5. **Diagnostica el doble conteo de `internet`** (dentro del IIF y como control).
6. **Diagnostica el look-ahead del PCA** (pesos estimados con 2020–2021).
7. **Compara cada cifra contra lo que dice el documento** y marca ✓ / ✗.
8. **Exporta todo a Markdown** (`RESULTADOS_CONSOLIDADO.md`).

Ejecutar: Kernel → Restart & Run All. Al final se genera el `.md`.

In [ ]:
# %pip install linearmodels

In [ ]:
#%pip install factor_analyzer

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# SECCIÓN 0 — Configuración
# ═══════════════════════════════════════════════════════════════════════════
import warnings
warnings.filterwarnings('ignore')

import sys
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA as SklearnPCA
from scipy import stats
from scipy.stats import chi2 as chi2_dist

from linearmodels.panel import PanelOLS, RandomEffects

try:
    from factor_analyzer.factor_analyzer import calculate_kmo, calculate_bartlett_sphericity
    FA_OK = True
except ImportError:
    FA_OK = False

pd.options.display.float_format = '{:.4f}'.format
np.random.seed(42)

# ── AJUSTAR AQUÍ ───────────────────────────────────────────────────────────
RUTA_DATOS = "../../data/legacy/panel_fintech_colombia_trimestral.xlsx"  # relativa a notebooks/legacy/
COL_TIEMPO = 'Fecha'
OUT_MD     = 'RESULTADOS_CONSOLIDADO.md'
# ───────────────────────────────────────────────────────────────────────────

# ── Logger: imprime y acumula para el Markdown ─────────────────────────────
_MD, _BLK = [], [False]

def _open():
    if not _BLK[0]:
        _MD.append('```text'); _BLK[0] = True

def _close():
    if _BLK[0]:
        _MD.append('```'); _BLK[0] = False

def P(*args, sep=' '):
    s = sep.join(str(a) for a in args)
    print(s)
    _open(); _MD.append(s)

def H(titulo, n=2):
    print('\n' + '=' * 78); print(titulo); print('=' * 78)
    _close(); _MD.append(''); _MD.append('#' * n + ' ' + titulo); _MD.append('')

def save_md(path=OUT_MD):
    _close()
    Path(path).write_text('\n'.join(_MD), encoding='utf-8')
    print(f'\n>>> Markdown escrito: {Path(path).resolve()}')

def stars(p):
    return '***' if p < 0.01 else ('**' if p < 0.05 else ('*' if p < 0.10 else ''))

H('SECCIÓN 0 — Entorno', 2)
P(f'pandas {pd.__version__} · numpy {np.__version__}')
P(f'factor_analyzer disponible: {FA_OK}')
P(f'Ruta de datos: {RUTA_DATOS}')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# SECCIÓN 1 — Constantes
# ═══════════════════════════════════════════════════════════════════════════
RENAME_MAP = {
    'Depto Base': 'departamento', 'Año': 'anio', 'Key': 'key',
    'Poblacion': 'poblacion', 'Capital': 'capital',
    'NRO CORRESPONSALES PROPIOS': 'nro_corresp_propios',
    'NRO CORRESPONSALES TERCERIZADOS': 'nro_corresp_terceros',
    'NRO CORRESPONSALES ACTIVOS': 'nro_corresp_activos',
    'NRO CORRESPONSALES TOTALES': 'nro_corresp_totales',
    'NRO DEPOSITOS': 'nro_depositos', 'MONTO DEPOSITOS': 'monto_depositos',
    'NRO GIROS ENVIADOS': 'nro_giros_env', 'MONTO GIROS ENVIADOS': 'monto_giros_env',
    'NRO GIROS RECIBIDOS': 'nro_giros_rec', 'MONTO GIROS RECIBIDOS': 'monto_giros_rec',
    'NRO PAGOS': 'nro_pagos', 'MONTO PAGOS': 'monto_pagos',
    'NRO RETIROS': 'nro_retiros', 'MONTO RETIROS': 'monto_retiros',
    'NRO TRANSFERENCIAS': 'nro_transf', 'MONTO TRANSFERENCIAS': 'monto_transf',
    'NRO TOTAL': 'nro_total', 'MONTO TOTAL': 'monto_total',
    'NRO CTA AHORRO HASTA 1 SMMLV': 'nro_cta_ah_hasta1smmlv',
    'SALDO CTA AHORRO HASTA 1 SMMLV': 'saldo_cta_ah_hasta1smmlv',
    'NRO CTA AHORRO > 1 SMMLV HASTA 3 SMMLV': 'nro_cta_ah_1a3smmlv',
    'SALDO CTA AHORRO> 1 SMMLV HASTA 3 SMMLV': 'saldo_cta_ah_1a3smmlv',
    'NRO CTA AHORRO> 3 SMMLV HASTA 5 SMMLV': 'nro_cta_ah_3a5smmlv',
    'SALDO CTA AHORRO> 3 SMMLV HASTA 5 SMMLV': 'saldo_cta_ah_3a5smmlv',
    'NRO CTA AHORRO ACTIVAS': 'nro_cta_ah_activas',
    'SALDO CTA AHORRO ACTIVAS': 'saldo_cta_ah_activas',
    'NRO CTA AHORRO MUJERES': 'nro_cta_ah_mujeres',
    'NRO CTA AHORRO HOMBRES': 'nro_cta_ah_hombres',
    'NRO TOTAL CTA AHORROS': 'nro_total_cta_ah',
    'SALDO TOTAL CTA AHORROS': 'saldo_total_cta_ah',
    'NRO CTA AHORRO ELECTRONICAS ACTIVAS': 'nro_cta_elect_activas',
    'SALDO CTA AHORRO ELECTRONICAS ACTIVAS': 'saldo_cta_elect_activas',
    'NRO CTA AHORRO ELECTRONICAS MUJERES': 'nro_cta_elect_mujeres',
    'NRO CTA AHORRO ELECTRONICAS HOMBRES': 'nro_cta_elect_hombres',
    'NRO TOTAL CTA AHORROS ELECTRONICAS': 'nro_total_cta_elect',
    'SALDO TOTAL CTA AHORROS ELECTRONICAS': 'saldo_total_cta_elect',
    'NRO CREDITO CONSUMO MUJERES': 'nro_cred_cons_mujeres',
    'MONTO CREDITO CONSUMO MUJERES': 'monto_cred_cons_mujeres',
    'NRO CREDITO CONSUMO HOMBRES': 'nro_cred_cons_hombres',
    'MONTO CREDITO CONSUMO HOMBRES': 'monto_cred_cons_hombres',
    'NRO TOTAL CREDITO CONSUMO': 'nro_total_cred_cons',
    'MONTO TOTAL CREDITO CONSUMO': 'monto_total_cred_cons',
    'NRO CREDITO VIVIENDA MUJERES': 'nro_cred_viv_mujeres',
    'MONTO CREDITO VIVIENDA MUJERES': 'monto_cred_viv_mujeres',
    'NRO CREDITO VIVIENDA HOMBRES': 'nro_cred_viv_hombres',
    'MONTO CREDITO VIVIENDA HOMBRES': 'monto_cred_viv_hombres',
    'NRO TOTAL CREDITO VIVIENDA': 'nro_total_cred_viv',
    'MONTO TOTAL CREDITO VIVIENDA': 'monto_total_cred_viv',
    'NRO MICROCREDITO HASTA 1 SMMLV': 'nro_micro_hasta1smmlv',
    'MONTO MICROCREDITO HASTA 1 SMMLV': 'monto_micro_hasta1smmlv',
    'NRO MICROCREDITO MUJERES': 'nro_micro_mujeres',
    'MONTO MICROCREDITO MUJERES': 'monto_micro_mujeres',
    'NRO MICROCREDITO HOMBRES': 'nro_micro_hombres',
    'MONTO MICROCREDITO HOMBRES': 'monto_micro_hombres',
    'NRO TOTAL MICROCREDITO': 'nro_total_micro',
    'MONTO TOTAL MICROCREDITO': 'monto_total_micro',
    'Total General Hogar Internet': 'hogares_internet_total',
    'Conexión a internet %': 'internet_pct',
    'Fijo %': 'internet_fijo_pct', 'Movil %': 'internet_movil_pct',
    'Fijo Movil %': 'internet_fijo_movil_pct',
    'Años promedio de educación': 'educacion_anios',
    'IPC %': 'IPC_pct', 'Empleo Formal': 'Empleo_Formal',
    'Empleo Informal': 'Empleo_Informal', 'Superficie Km2': 'Superficie_km2',
    'Densidad Poblacional': 'Densidad_Poblacional',
    'PIB Contribucion Al Total': 'pib_contribucion_pct',
    'PIB Percapita': 'pib_percapita', 'PIB Crecimiento': 'pib_crecimiento',
}

VARS_PCA = ['d_acc_corresp', 'd_acc_cta_ah', 'd_acc_internet',
            'd_uso_pagos_pc', 'd_uso_transf_pc', 'd_uso_depositos_pc',
            'd_pro_micro', 'd_pro_cred_cons', 'd_pro_cred_viv']

# Variante sin internet, para el diagnóstico de doble conteo
VARS_PCA_SIN_INTERNET = [v for v in VARS_PCA if v != 'd_acc_internet']

LABELS_PCA = {
    'd_acc_corresp': 'Corresponsales', 'd_acc_cta_ah': 'Cuentas ahorro',
    'd_acc_internet': 'Internet (%)', 'd_uso_pagos_pc': 'Pagos digitales',
    'd_uso_transf_pc': 'Transferencias', 'd_uso_depositos_pc': 'Depósitos',
    'd_pro_micro': 'Microcrédito/PIB', 'd_pro_cred_cons': 'Créd. consumo/PIB',
    'd_pro_cred_viv': 'Créd. vivienda/PIB',
}

CONTROLES = 'educacion + internet + densidad + informalidad'

REGION_MAP = {
    'atlantico': 1, 'atlántico': 1, 'bolivar': 1, 'bolívar': 1,
    'magdalena': 1, 'la guajira': 1, 'guajira': 1, 'cesar': 1,
    'cordoba': 1, 'córdoba': 1, 'sucre': 1, 'san andres': 1,
    'san andrés': 1, 'san andres,prov y santa catalina': 1,
    'antioquia': 0, 'boyaca': 0, 'boyacá': 0, 'caldas': 0,
    'cundinamarca': 0, 'bogota': 0, 'bogotá': 0,
    'bogota d.c.': 0, 'bogotá d.c.': 0,
    'norte de santander': 0, 'santander': 0, 'risaralda': 0,
    'quindio': 0, 'quindío': 0, 'tolima': 0, 'huila': 0,
    'choco': 2, 'chocó': 2, 'valle del cauca': 2, 'valle': 2,
    'cauca': 2, 'narino': 2, 'nariño': 2,
    'arauca': 3, 'casanare': 3, 'meta': 3, 'vichada': 3,
    'amazonas': 4, 'caqueta': 4, 'caquetá': 4, 'guainia': 4,
    'guainía': 4, 'guaviare': 4, 'putumayo': 4, 'vaupes': 4, 'vaupés': 4,
}
REGION_LABEL = {0: 'Andina', 1: 'Caribe', 2: 'Pacífica', 3: 'Orinoquía', 4: 'Amazonía'}

# ── Cifras tal como aparecen en el DOCUMENTO (para la verificación final) ──
DOC = {
    'A1': dict(beta=31.1133, se=6.6263, N=396, r2=0.0488),
    'A2': dict(beta=62.4924, se=8.3614, N=363, r2=0.1247),
    'A3': dict(beta=-0.1470, se=7.2027, N=363, r2=0.0077),
    'A4': dict(beta=None,    se=None,   N=None, r2=None),
    'B1': dict(beta=5.7043,  se=5.5871, N=429, r2=0.0906),
    'B2': dict(beta=35.5537, se=9.5553, N=429, r2=0.1992),   # <- tabla; texto dice 5.96
    'B3': dict(beta=28.5079, se=4.8183, N=396, r2=0.1946),
    'B4': dict(beta=-4.9256, se=4.3580, N=396, r2=0.1748),
}
DOC_OTROS = {
    'KMO': 0.7189, 'Bartlett_chi2': 2543.45, 'var_acum_4pc': 84.4,
    'Hausman_chi2': 33.93, 'Mundlak_IIF_mean': 4.11,
    'corr_between': 0.094, 'corr_within': -0.164,
    'preCOVID_beta': 0.12, 'preCOVID_se': 1.57, 'preCOVID_r2': 0.576,
    'preCOVID_2way_beta': -3.94, 'preCOVID_2way_se': 6.03,
    'Pesaran_CD': 66.81, 'DK_se_de': 5.77, 'DK_se_a': 38.51,
    'trim_beta': 25.21, 'trim_N': 420, 'wins_beta': 23.31, 'wins_N': 396,
    'beta_log': 5.344, 'se_log': 1.626, 'N_log': 396, 'r2_log': 0.167,
    'rho_nickell': 0.4279, 'sesgo_nickell': -0.1098,
    'corr_orient': 0.8684,
}

VERIF = []   # se llena a lo largo del notebook

def chk(nombre, obtenido, documento, tol=0.02, nota=''):
    """Compara un valor obtenido contra el que aparece en el documento."""
    if documento is None:
        estado = 'SIN DATO EN DOC'
    elif obtenido is None:
        estado = 'NO ESTIMADO'
    else:
        rel = abs(obtenido - documento) / max(abs(documento), 1e-9)
        estado = 'OK' if rel <= tol else 'DISCREPA'
    VERIF.append(dict(item=nombre, obtenido=obtenido, documento=documento,
                      estado=estado, nota=nota))
    return estado

P('✓ Constantes cargadas.')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# SECCIÓN 2 — Carga de datos
# ═══════════════════════════════════════════════════════════════════════════
H('SECCIÓN 2 — Carga de datos')

panel_raw = pd.read_excel(RUTA_DATOS)
panel_raw.columns = panel_raw.columns.str.strip()
panel_raw = panel_raw.rename(columns=RENAME_MAP)

P(f'Dimensiones   : {panel_raw.shape}')
P(f'Departamentos : {panel_raw["departamento"].nunique()}')
P(f'Años          : {sorted(panel_raw["anio"].unique())}')
P(f'Columna tiempo: {COL_TIEMPO} — {panel_raw[COL_TIEMPO].nunique()} valores únicos')
P(f'Períodos      : {sorted(panel_raw[COL_TIEMPO].unique())[:4]} ... '
  f'{sorted(panel_raw[COL_TIEMPO].unique())[-2:]}')
nas = panel_raw.isnull().sum()
nas = nas[nas > 0]
P(f'Variables con NA: {len(nas)}')

## DIAGNÓSTICO D1 — ¿El PIB es un valor anual repetido?

Esta es la pregunta abierta más importante. Si `pib_percapita` y `pib_crecimiento`
toman un solo valor por año dentro de cada departamento, entonces:

- la variable dependiente no tiene variación trimestral;
- `crec_pib_pc` (log-diff) será exactamente 0 en ~3 de cada 4 observaciones;
- `crec_pib(t-1) = crec_pib(t)` en ~3 de cada 4 observaciones, y el modelo dinámico
  estaría regresando la variable sobre sí misma.

In [ ]:
H('DIAGNÓSTICO D1 — Estructura temporal del PIB')

d = panel_raw.copy()
for col in ['pib_percapita', 'pib_crecimiento', 'IPC_pct',
            'internet_pct', 'educacion_anios', 'Empleo_Informal']:
    if col in d.columns:
        nu = d.groupby(['departamento', 'anio'])[col].nunique()
        P(f'{col:<22} valores distintos por (depto, año): '
          f'min={nu.min()}  max={nu.max()}  media={nu.mean():.2f}')

P('')
P('Lectura: si max = 1 la variable es ANUAL repetida en los 4 trimestres.')
P('         si max = 4 la variable varía trimestralmente.')

P('')
nu_dep = panel_raw.groupby('departamento')['pib_percapita'].nunique()
P(f'pib_percapita — valores distintos por departamento (sobre 14 trimestres):')
P(f'  min={nu_dep.min()}  max={nu_dep.max()}  media={nu_dep.mean():.2f}')
P(f'  (5 ≈ un valor por año; 14 ≈ variación trimestral real)')

# fracción de transiciones idénticas en el crecimiento
tmp = panel_raw.sort_values(['departamento', COL_TIEMPO]).copy()
tmp['g_lag'] = tmp.groupby('departamento')['pib_crecimiento'].shift(1)
frac_ident = (tmp['pib_crecimiento'] == tmp['g_lag']).mean()
P('')
P(f'Fracción de observaciones con crec_pib(t) == crec_pib(t-1): {frac_ident:.3f}')
P('  → si ≈ 0.75, el modelo dinámico regresa la variable sobre sí misma en 3 de cada 4 casos.')

VERIF.append(dict(item='D1 pib_percapita distintos por depto',
                  obtenido=float(nu_dep.mean()), documento=None,
                  estado='DIAGNÓSTICO', nota='5≈anual repetido / 14≈trimestral'))
VERIF.append(dict(item='D1 fracción crec_pib(t)==crec_pib(t-1)',
                  obtenido=float(frac_ident), documento=None,
                  estado='DIAGNÓSTICO', nota='0.75 ⇒ dependiente sin variación trimestral'))

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# SECCIÓN 3 — Construcción de variables  (ordenado por FECHA, no por año)
# ═══════════════════════════════════════════════════════════════════════════
H('SECCIÓN 3 — Construcción de variables')

def construir_variables(df):
    df = df.copy()
    # Deflactor IPC base 2016 = 100
    ipc = (df[['anio', 'IPC_pct']].drop_duplicates('anio')
           .sort_values('anio').reset_index(drop=True))
    base = pd.DataFrame({'anio': [2016], 'IPC_pct': [0.0]})
    ipc = (pd.concat([base, ipc]).drop_duplicates('anio', keep='first')
           .sort_values('anio').reset_index(drop=True))
    ipc['ipc_index'] = 100 * (1 + ipc['IPC_pct'] / 100).cumprod()
    df['ipc_index'] = df['anio'].map(ipc.set_index('anio')['ipc_index'])

    # FIX: ordenar por departamento y FECHA (los notebooks previos ordenaban por 'anio')
    df = df.sort_values(['departamento', COL_TIEMPO]).reset_index(drop=True)

    pob = df['poblacion']
    P_ = df['ipc_index'] / 100

    df['pib_pc_real']       = df['pib_percapita'] / P_
    df['pib_nominal_aprox'] = df['pib_percapita'] * pob
    df['crec_pib']          = df['pib_crecimiento']
    df['crec_pib_pc'] = (df.groupby('departamento')['pib_pc_real']
                           .transform(lambda x: (np.log(x) - np.log(x.shift(1))) * 100))

    df['fintech_nro_pc'] = (df['nro_total'] / pob) * 1_000
    df['log_fintech_pc'] = np.log(df['fintech_nro_pc'].clip(lower=1e-9))

    df['d_acc_corresp']    = (df['nro_corresp_activos'] / pob) * 10_000
    df['d_acc_cta_ah']     = (df['nro_total_cta_ah'] / pob) * 1_000
    df['d_acc_internet']   = df['internet_pct']
    df['d_uso_pagos_pc']   = (df['nro_pagos'] / pob) * 1_000
    df['d_uso_transf_pc']  = (df['nro_transf'] / pob) * 1_000
    df['d_uso_depositos_pc'] = (df['nro_depositos'] / pob) * 1_000
    df['d_pro_micro']      = df['monto_total_micro'] / df['pib_nominal_aprox']
    df['d_pro_cred_cons']  = df['monto_total_cred_cons'] / df['pib_nominal_aprox']
    df['d_pro_cred_viv']   = df['monto_total_cred_viv'] / df['pib_nominal_aprox']

    df['densidad']  = df['Densidad_Poblacional']
    df['educacion'] = df['educacion_anios']
    df['informalidad'] = (df['Empleo_Informal'] /
                          (df['Empleo_Formal'] + df['Empleo_Informal'])) * 100
    df['d_covid'] = (df['anio'] == 2020).astype(int)

    med = df.groupby('departamento')['densidad'].median()
    df['densidad_mediana'] = df['departamento'].map(med)
    umbral = df['densidad_mediana'].quantile(0.75)
    df['urbano'] = (df['densidad_mediana'] > umbral).astype(int)
    df['zona'] = df['urbano'].map({1: 'Urbano', 0: 'Rural'})
    df['d_region'] = df['departamento'].str.strip().str.lower().map(REGION_MAP)
    df['t_idx'] = df.groupby('departamento').cumcount() + 1

    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    return df, umbral

panel, umbral_p75 = construir_variables(panel_raw)
panel = panel[panel['anio'].between(2017, 2021)].reset_index(drop=True)

P(f'Obs totales            : {len(panel)}')
P(f'Umbral p75 densidad    : {umbral_p75:.1f} hab/km²')
P(f'Urbanos / Rurales      : {panel[panel.urbano==1].departamento.nunique()} / '
  f'{panel[panel.urbano==0].departamento.nunique()}')
P(f'crec_pib     no nulos  : {panel["crec_pib"].notna().sum()}')
P(f'crec_pib_pc  no nulos  : {panel["crec_pib_pc"].notna().sum()}')
P(f'Regiones sin mapear    : {panel["d_region"].isna().sum()} obs')
if panel['d_region'].isna().any():
    P(f'  departamentos: {sorted(panel.loc[panel.d_region.isna(),"departamento"].unique())}')

# D1 (continuación): ceros exactos en la dependiente per cápita
frac_cero = (panel['crec_pib_pc'].round(6) == 0).mean()
P('')
P(f'Fracción de crec_pib_pc exactamente igual a 0: {frac_cero:.3f}')
P('  → si ≈ 0.75, la dependiente per cápita no varía dentro del año.')
VERIF.append(dict(item='D1 fracción crec_pib_pc == 0', obtenido=float(frac_cero),
                  documento=None, estado='DIAGNÓSTICO',
                  nota='0.75 ⇒ log-diff nulo dentro del año'))

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# SECCIÓN 4 — PCA e Índice de Inclusión Financiera
# ═══════════════════════════════════════════════════════════════════════════
H('SECCIÓN 4 — PCA e IIF')

def construir_iif(df, vars_pca, id_cols, sign_ref='d_acc_cta_ah',
                  umbral_var=80.0, etiqueta='IIF', verbose=True,
                  fit_mask=None):
    """
    Construye IIF_Multidim (ponderado por varianza de los componentes retenidos)
    e IIF_PC1.  Si fit_mask se pasa, el PCA se AJUSTA solo sobre esas filas
    (para el diagnóstico de look-ahead) y se aplica a todo el panel.
    """
    datos = df[id_cols + vars_pca].dropna().copy()
    X_raw = datos[vars_pca].values

    kmo_model, chi2_b, p_b = np.nan, np.nan, np.nan
    if FA_OK and verbose:
        _, kmo_model = calculate_kmo(X_raw)
        chi2_b, p_b = calculate_bartlett_sphericity(X_raw)
        P(f'[{etiqueta}] KMO = {kmo_model:.4f} | Bartlett χ² = {chi2_b:.2f} (p = {p_b:.4f})')

    scaler = StandardScaler()
    if fit_mask is not None:
        m = fit_mask.loc[datos.index] if hasattr(fit_mask, 'loc') else fit_mask
        scaler.fit(X_raw[m.values])
        X_sc = scaler.transform(X_raw)
        pca = SklearnPCA(); pca.fit(X_sc[m.values])
    else:
        X_sc = scaler.fit_transform(X_raw)
        pca = SklearnPCA(); pca.fit(X_sc)

    scores  = pca.transform(X_sc)
    var_pct = pca.explained_variance_ratio_ * 100
    var_ac  = np.cumsum(var_pct)
    n_comp  = int(np.searchsorted(var_ac, umbral_var)) + 1

    if verbose:
        P(f'[{etiqueta}] Componentes retenidos (≥{umbral_var}%): {n_comp}')
        for i, (v, a) in enumerate(zip(var_pct, var_ac)):
            P(f'   PC{i+1}: {v:5.2f}%  (acum {a:6.2f}%)' + (' ◄' if i < n_comp else ''))

    ret = scores[:, :n_comp].copy()
    comps = pca.components_.copy()
    for k in range(n_comp):
        if np.corrcoef(ret[:, k], datos[sign_ref])[0, 1] < 0:
            ret[:, k] = -ret[:, k]; comps[k] = -comps[k]

    EPS = 1e-4
    pc1 = ret[:, 0]
    datos['IIF_PC1'] = (pc1 - pc1.min()) / (pc1.max() - pc1.min()) + EPS

    pesos = var_pct[:n_comp] / var_pct[:n_comp].sum()
    raw   = ret @ pesos
    iif   = (raw - raw.min()) / (raw.max() - raw.min()) + EPS
    datos['IIF'] = iif
    datos['log_IIF'] = np.log(iif)
    datos['IIF_std'] = (iif - iif.mean()) / iif.std()

    corr_orient = np.corrcoef(datos['IIF'], datos[sign_ref])[0, 1]
    if verbose:
        P(f'[{etiqueta}] Pesos de los {n_comp} componentes: '
          f'{np.round(pesos, 4).tolist()}')
        P(f'[{etiqueta}] Corr(IIF, cuentas ahorro pc) = {corr_orient:.4f}')

    loadings = pd.DataFrame(comps[:n_comp].T,
                            index=[LABELS_PCA.get(v, v) for v in vars_pca],
                            columns=[f'PC{i+1} ({var_pct[i]:.1f}%)' for i in range(n_comp)])
    info = dict(kmo=kmo_model, bartlett=chi2_b, p_bartlett=p_b,
                n_comp=n_comp, var_pct=var_pct, var_ac=var_ac,
                pesos=pesos, loadings=loadings, corr_orient=corr_orient)
    return datos, info

ID_COLS = ['departamento', COL_TIEMPO]
iif_df, info_iif = construir_iif(panel, VARS_PCA, ID_COLS, etiqueta='IIF principal')

panel = panel.merge(iif_df[ID_COLS + ['IIF', 'IIF_PC1', 'log_IIF', 'IIF_std']],
                    on=ID_COLS, how='left')

P('')
P('Cargas factoriales (componentes retenidos):')
P(info_iif['loadings'].round(4).to_string())

chk('KMO', float(info_iif['kmo']) if info_iif['kmo'] == info_iif['kmo'] else None,
    DOC_OTROS['KMO'], tol=0.01)
chk('Bartlett chi2', float(info_iif['bartlett']) if info_iif['bartlett'] == info_iif['bartlett'] else None,
    DOC_OTROS['Bartlett_chi2'], tol=0.01)
chk('Varianza acumulada 4 PC', float(info_iif['var_ac'][3]), DOC_OTROS['var_acum_4pc'], tol=0.01)
chk('Corr orientación IIF~cuentas', float(info_iif['corr_orient']),
    DOC_OTROS['corr_orient'], tol=0.03)

## DIAGNÓSTICO D2 — `internet` está dentro del IIF y también es control

`d_acc_internet` entra al PCA (Tabla 3 de la tesis) y `internet` entra como control
estructural en todas las regresiones. Aquí se construye un IIF sin internet para
ver si el resultado depende de ese doble conteo.

In [ ]:
H('DIAGNÓSTICO D2 — IIF sin internet')

iif_sin_net, info_sin_net = construir_iif(panel, VARS_PCA_SIN_INTERNET, ID_COLS,
                                          etiqueta='IIF sin internet')
panel = panel.merge(
    iif_sin_net[ID_COLS + ['IIF']].rename(columns={'IIF': 'IIF_sin_net'}),
    on=ID_COLS, how='left')
P('')
P(f'Corr(IIF, IIF_sin_net) = {panel[["IIF","IIF_sin_net"]].corr().iloc[0,1]:.4f}')

## DIAGNÓSTICO D3 — Look-ahead del PCA

El notebook de correcciones identificó este riesgo y no logró correr la verificación.
Aquí se ajusta el PCA **solo con 2017–2019** y se aplican esos pesos a todo el panel.

In [ ]:
H('DIAGNÓSTICO D3 — PCA out-of-sample (pesos de 2017–2019)')

try:
    mask_pre = panel['anio'].between(2017, 2019)
    base = panel[ID_COLS + VARS_PCA + ['anio']].dropna().reset_index(drop=True)
    mask_fit = base['anio'].between(2017, 2019)
    iif_oos, info_oos = construir_iif(base, VARS_PCA, ID_COLS,
                                      etiqueta='IIF out-of-sample',
                                      fit_mask=mask_fit)
    panel = panel.merge(
        iif_oos[ID_COLS + ['IIF']].rename(columns={'IIF': 'IIF_oos'}),
        on=ID_COLS, how='left')
    P('')
    P(f'Corr(IIF, IIF_oos) = {panel[["IIF","IIF_oos"]].corr().iloc[0,1]:.4f}')
    P('  → si la correlación es alta (>0.95) el look-ahead no altera el ordenamiento.')
except Exception as e:
    P(f'⚠ D3 falló: {e}')
    panel['IIF_oos'] = np.nan

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# SECCIÓN 5 — Panel econométrico  ★ REZAGOS UNA SOLA VEZ, PANEL COMPLETO ★
# ═══════════════════════════════════════════════════════════════════════════
H('SECCIÓN 5 — Panel econométrico (construcción corregida de rezagos)')

pm = panel.copy().rename(columns={'d_acc_internet': 'internet'})
pm = pm.sort_values(['departamento', COL_TIEMPO]).reset_index(drop=True)

VARS_LAG = ['crec_pib', 'crec_pib_pc', 'IIF', 'log_IIF', 'IIF_std', 'IIF_PC1',
            'IIF_sin_net', 'IIF_oos', 'log_fintech_pc']
for v in VARS_LAG:
    if v in pm.columns:
        for L in (1, 2):
            pm[f'{v}_lag{L}'] = pm.groupby('departamento')[v].shift(L)

pm['IIF_mean']   = pm.groupby('departamento')['IIF'].transform('mean')
pm['IIF_within'] = pm['IIF'] - pm['IIF_mean']
pm['IIF_lag1_mean']   = pm.groupby('departamento')['IIF_lag1'].transform('mean')
pm['IIF_lag1_within'] = pm['IIF_lag1'] - pm['IIF_lag1_mean']

pm = pm.set_index(['departamento', COL_TIEMPO])

P(f'Entidades : {pm.index.get_level_values(0).nunique()}')
P(f'Períodos  : {pm.index.get_level_values(1).nunique()}')
P(f'Obs base  : {len(pm)}')
P('')
P('Trazabilidad de la muestra (todos los rezagos vienen del panel de 462):')
for etiqueta, cols in [
    ('crec_pib + IIF                       ', ['crec_pib', 'IIF']),
    ('crec_pib + IIF_lag1 + crec_pib_lag1  ', ['crec_pib', 'IIF_lag1', 'crec_pib_lag1']),
    ('crec_pib_pc + IIF                    ', ['crec_pib_pc', 'IIF']),
    ('crec_pib_pc + IIF_lag1 + lag DV      ', ['crec_pib_pc', 'IIF_lag1', 'crec_pib_pc_lag1']),
    ('crec_pib_pc + IIF_lag2 + lag DV      ', ['crec_pib_pc', 'IIF_lag2', 'crec_pib_pc_lag1']),
]:
    cols_ok = [c for c in cols if c in pm.columns] + \
              ['internet', 'educacion', 'densidad', 'informalidad']
    P(f'  {etiqueta}: {len(pm.dropna(subset=cols_ok)):>4} obs')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# SECCIÓN 6 — Estimadores
# ═══════════════════════════════════════════════════════════════════════════
CTRL = ['internet', 'educacion', 'densidad', 'informalidad']

def fe(data, var_iif, dv, dinamico=True, time_effects=False, label='',
       cov='clustered', extra=None):
    """FE de entidad, opcionalmente con TimeEffects y/o rezago de la dependiente."""
    dv_lag = f'{dv}_lag1'
    need = [dv, var_iif] + CTRL + ([dv_lag] if dinamico else []) + (extra or [])
    need = [c for c in need if c in data.columns]
    sub = data.dropna(subset=need)
    if len(sub) < 30:
        P(f'  {label}: muestra insuficiente ({len(sub)})'); return None
    partes = ([dv_lag] if dinamico else []) + [var_iif] + CTRL + (extra or [])
    formula = f'{dv} ~ ' + ' + '.join(partes) + ' + EntityEffects' + \
              (' + TimeEffects' if time_effects else '')
    try:
        mod = PanelOLS.from_formula(formula, data=sub, drop_absorbed=True)
        if cov == 'driscoll-kraay':
            m = mod.fit(cov_type='kernel')
        else:
            m = mod.fit(cov_type='clustered', cluster_entity=True)
        if label:
            b, se, p = m.params[var_iif], m.std_errors[var_iif], m.pvalues[var_iif]
            P(f'  {label:<48} β={b:+9.4f}{stars(p):<3} SE={se:7.4f} '
              f'p={p:.4f} N={m.nobs:>4.0f} R²={m.rsquared:.4f}')
        return m
    except Exception as e:
        P(f'  {label}: ERROR {e}'); return None

def row(m, var):
    if m is None:
        return dict(beta=None, se=None, p=None, N=None, r2=None)
    return dict(beta=float(m.params[var]), se=float(m.std_errors[var]),
                p=float(m.pvalues[var]), N=float(m.nobs), r2=float(m.rsquared))

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# SECCIÓN 7 — TABLA 6 CONSOLIDADA  (incluye A4, que faltaba)
# ═══════════════════════════════════════════════════════════════════════════
H('SECCIÓN 7 — TABLA 6 consolidada')

P('BLOQUE A — DV: crecimiento del PIB per cápita')
mA1 = fe(pm, 'IIF',      'crec_pib_pc', dinamico=False, label='A1 FE-Est   | IIF')
mA2 = fe(pm, 'IIF_lag1', 'crec_pib_pc', dinamico=True,  label='A2 FE-Dyn   | IIF_lag1')
mA3 = fe(pm, 'IIF_lag1', 'crec_pib_pc', dinamico=True,  time_effects=True,
         label='A3 2way FE  | IIF_lag1  ← REFERENCIA')
mA4 = fe(pm, 'IIF',      'crec_pib_pc', dinamico=False, time_effects=True,
         label='A4 2way FE  | IIF       ← FALTABA')

P('')
P('BLOQUE B — DV: crecimiento del PIB agregado')
mB1 = fe(pm, 'IIF',          'crec_pib', dinamico=False, label='B1 FE-Est   | IIF')
mB2 = fe(pm, 'IIF_lag1',     'crec_pib', dinamico=True,  label='B2 FE-Dyn   | IIF_lag1')
mB3 = fe(pm, 'IIF_PC1_lag1', 'crec_pib', dinamico=True,  label='B3 FE-Dyn   | IIF_PC1_lag1')
mB4 = fe(pm, 'IIF_lag1',     'crec_pib', dinamico=True,  time_effects=True,
         label='B4 2way FE  | IIF_lag1')

RES = {'A1': row(mA1, 'IIF'), 'A2': row(mA2, 'IIF_lag1'), 'A3': row(mA3, 'IIF_lag1'),
       'A4': row(mA4, 'IIF'), 'B1': row(mB1, 'IIF'), 'B2': row(mB2, 'IIF_lag1'),
       'B3': row(mB3, 'IIF_PC1_lag1'), 'B4': row(mB4, 'IIF_lag1')}

P('')
P('Comparación entity vs two-way SOBRE LA MISMA MUESTRA:')
for blq, var, m_ent, m_2w in [('A (per cápita)', 'IIF_lag1', mA2, mA3),
                              ('B (agregado)  ', 'IIF_lag1', mB2, mB4)]:
    if m_ent is not None and m_2w is not None:
        P(f'  Bloque {blq}: entity β={m_ent.params[var]:+8.4f} (N={m_ent.nobs:.0f}) → '
          f'two-way β={m_2w.params[var]:+8.4f} (N={m_2w.nobs:.0f})  '
          f'| ΔR² = {m_2w.rsquared - m_ent.rsquared:+.4f}')

for k in RES:
    chk(f'Tabla6 {k} β', RES[k]['beta'], DOC[k]['beta'], tol=0.02)
    chk(f'Tabla6 {k} SE', RES[k]['se'],  DOC[k]['se'],  tol=0.02)
    chk(f'Tabla6 {k} N',  RES[k]['N'],   DOC[k]['N'],   tol=0.001)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# SECCIÓN 8 — Hausman, Mundlak, between/within
# ═══════════════════════════════════════════════════════════════════════════
H('SECCIÓN 8 — Hausman, Mundlak, descomposición')

m_fe = None   # referenciado también en 10b y 10e
sub_h = pm.dropna(subset=['crec_pib', 'IIF_lag1', 'crec_pib_lag1'] + CTRL)
try:
    f_h = 'crec_pib ~ crec_pib_lag1 + IIF_lag1 + ' + ' + '.join(CTRL)
    m_fe = PanelOLS.from_formula(f_h + ' + EntityEffects', data=sub_h).fit(
        cov_type='clustered', cluster_entity=True)
    m_re = RandomEffects.from_formula(f_h, data=sub_h).fit(
        cov_type='clustered', cluster_entity=True)
    comunes = [v for v in m_fe.params.index if v in m_re.params.index]
    diff = m_fe.params[comunes].values - m_re.params[comunes].values
    Vd = m_fe.cov.loc[comunes, comunes].values - m_re.cov.loc[comunes, comunes].values
    Hst = float(diff @ np.linalg.pinv(Vd) @ diff)
    p_h = 1 - chi2_dist.cdf(Hst, df=len(comunes))
    P(f'Hausman: χ²({len(comunes)}) = {Hst:.4f}, p = {p_h:.4f}, N = {m_fe.nobs:.0f}')
    chk('Hausman chi2', Hst, DOC_OTROS['Hausman_chi2'], tol=0.05)
except Exception as e:
    P(f'Hausman: ERROR {e}')

try:
    f_m = ('crec_pib ~ crec_pib_lag1 + IIF_lag1_within + IIF_lag1_mean + '
           + ' + '.join(CTRL))
    m_mu = RandomEffects.from_formula(f_m, data=sub_h).fit(
        cov_type='clustered', cluster_entity=True)
    bw, bm = m_mu.params['IIF_lag1_within'], m_mu.params['IIF_lag1_mean']
    P(f'Mundlak: β_within = {bw:+.4f} (p={m_mu.pvalues["IIF_lag1_within"]:.4f}) | '
      f'β_mean = {bm:+.4f} (p={m_mu.pvalues["IIF_lag1_mean"]:.4f})')
    chk('Mundlak IIF_mean', float(bm), DOC_OTROS['Mundlak_IIF_mean'], tol=0.05,
        nota='doc=4.11 | inicial=1.85 | correcciones=6.77')
except Exception as e:
    P(f'Mundlak: ERROR {e}')

ent = sub_h.index.get_level_values(0)
ib = sub_h['IIF'].groupby(ent).transform('mean')
pb = sub_h['crec_pib'].groupby(ent).transform('mean')
cb = float(np.corrcoef(ib, pb)[0, 1])
cw = float(np.corrcoef(sub_h['IIF'] - ib, sub_h['crec_pib'] - pb)[0, 1])
P(f'Corr between = {cb:+.4f} | Corr within = {cw:+.4f}')
chk('Corr between', cb, DOC_OTROS['corr_between'], tol=0.10)
chk('Corr within',  cw, DOC_OTROS['corr_within'],  tol=0.10)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# SECCIÓN 9 — Sesgo de Nickell (ρ de AMBOS modelos)
# ═══════════════════════════════════════════════════════════════════════════
H('SECCIÓN 9 — Sesgo de Nickell')

T = pm.index.get_level_values(1).nunique()
for nombre, m, dvlag in [('PIB agregado ', mB2, 'crec_pib_lag1'),
                         ('PIB per cápita', mA2, 'crec_pib_pc_lag1')]:
    if m is not None and dvlag in m.params.index:
        rho = float(m.params[dvlag])
        sesgo = -(1 + rho) / (T - 1)
        P(f'{nombre}: ρ̂ = {rho:+.4f} | sesgo ≈ {sesgo:+.4f} | '
          f'ρ corregido ≈ {rho - sesgo:+.4f} | {abs(sesgo/rho)*100:5.1f}% de ρ̂')
        if 'agregado' in nombre:
            chk('rho Nickell (agregado)', rho, DOC_OTROS['rho_nickell'], tol=0.03,
                nota='el documento usa el ρ del modelo AGREGADO')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# SECCIÓN 10 — Robustez
# ═══════════════════════════════════════════════════════════════════════════
H('SECCIÓN 10 — Robustez')

P('[10a] Submuestra pre-pandemia 2017–2019')
pre = pm[pm['anio'].between(2017, 2019)]
m_pre  = fe(pre, 'IIF_lag1', 'crec_pib', label='Pre-COVID | FE Entity')
m_pre2 = fe(pre, 'IIF_lag1', 'crec_pib', time_effects=True, label='Pre-COVID | two-way')
if m_pre is not None:
    chk('Pre-COVID β', float(m_pre.params['IIF_lag1']), DOC_OTROS['preCOVID_beta'], tol=0.30)
    chk('Pre-COVID R²', float(m_pre.rsquared), DOC_OTROS['preCOVID_r2'], tol=0.05)

P('')
P('[10b] Pesaran CD (dependencia transversal)')
try:
    if m_fe is None:
        raise RuntimeError('m_fe no disponible (falló la Sección 8)')
    resid = m_fe.resids.unstack(level=0)
    R = resid.corr()
    N_, T_ = R.shape[0], resid.notna().sum().mean()
    iu = np.triu_indices_from(R.values, k=1)
    CD = np.sqrt(2 * T_ / (N_ * (N_ - 1))) * np.nansum(R.values[iu])
    p_cd = 2 * (1 - stats.norm.cdf(abs(CD)))
    P(f'CD = {CD:.4f}, p = {p_cd:.4f}  (N={N_}, T medio={T_:.1f})')
    chk('Pesaran CD', float(CD), DOC_OTROS['Pesaran_CD'], tol=0.05)
except Exception as e:
    P(f'Pesaran CD: ERROR {e}')

P('')
P('[10c] Driscoll-Kraay')
m_dk = fe(pm, 'IIF_lag1', 'crec_pib', cov='driscoll-kraay',
          label='FE Entity | DK SE')
if m_dk is not None and mB2 is not None:
    P(f'  SE clustered = {mB2.std_errors["IIF_lag1"]:.4f}  →  '
      f'DK = {m_dk.std_errors["IIF_lag1"]:.4f}')

P('')
P('[10d] Outliers sobre la dependiente')
for etiqueta, dfsub in [
    ('Trimming 1%',
     pm[pm['crec_pib'].between(pm['crec_pib'].quantile(0.01),
                               pm['crec_pib'].quantile(0.99))]),
    ('Winsorización 5%', pm.assign(crec_pib=pm['crec_pib'].clip(
        pm['crec_pib'].quantile(0.05), pm['crec_pib'].quantile(0.95)))),
]:
    m_o = fe(dfsub, 'IIF_lag1', 'crec_pib', label=etiqueta)

P('')
P('[10e] Autocorrelación serial de los residuos (Wooldridge)')
try:
    if m_fe is None:
        raise RuntimeError('m_fe no disponible (falló la Sección 8)')
    r = m_fe.resids.to_frame('e')
    r['e_lag'] = r.groupby(level=0)['e'].shift(1)
    rr = r.dropna()
    rho_e, p_e = stats.pearsonr(rr['e'], rr['e_lag'])
    P(f'r(e_t, e_t-1) = {rho_e:.4f}, p = {p_e:.4f}')
except Exception as e:
    P(f'Wooldridge: ERROR {e}')

P('')
P('[10f] Especificaciones alternativas del índice y del rezago')
fe(pm, 'IIF_lag2',         'crec_pib_pc', label='IIF_lag2      | PIB pc')
fe(pm, 'IIF_lag2',         'crec_pib_pc', time_effects=True, label='IIF_lag2 2way | PIB pc')
fe(pm, 'IIF_sin_net_lag1', 'crec_pib_pc', label='IIF sin internet | PIB pc  (D2)')
fe(pm, 'IIF_sin_net_lag1', 'crec_pib_pc', time_effects=True,
   label='IIF sin internet 2way | PIB pc  (D2)')
if pm['IIF_oos'].notna().any():
    fe(pm, 'IIF_oos_lag1', 'crec_pib_pc', label='IIF out-of-sample | PIB pc (D3)')
    fe(pm, 'IIF_oos_lag1', 'crec_pib_pc', time_effects=True,
       label='IIF out-of-sample 2way | PIB pc (D3)')

P('')
P('[10g] Tendencias lineales por departamento (alternativa a TimeEffects)')
pm_t = pm.copy()
m_tr = fe(pm_t, 'IIF_lag1', 'crec_pib', extra=['t_idx'],
          label='FE Entity + tendencia t_idx')

P('')
P('[10h] Semi-elasticidad logarítmica')
m_log = fe(pm, 'log_IIF_lag1', 'crec_pib_pc', label='log(IIF)_lag1 | PIB pc')
if m_log is not None:
    b = float(m_log.params['log_IIF_lag1'])
    P(f'  β_log = {b:.4f} → +1% IIF = {b*np.log(1.01):.4f} pp | '
      f'+10% IIF = {b*np.log(1.10):.4f} pp')
    chk('β semi-elasticidad log', b, DOC_OTROS['beta_log'], tol=0.02)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# SECCIÓN 11 — Heterogeneidad regional (Tabla 7)
# ═══════════════════════════════════════════════════════════════════════════
H('SECCIÓN 11 — Heterogeneidad regional')

filas, total = [], 0
for cod, nombre in REGION_LABEL.items():
    dr = pm[pm['d_region'] == cod]
    m_r = fe(dr, 'IIF_lag1', 'crec_pib', label=f'{nombre}')
    if m_r is None:
        continue
    n_dep = dr.dropna(subset=['crec_pib', 'IIF_lag1', 'crec_pib_lag1'] + CTRL
                      ).index.get_level_values(0).nunique()
    filas.append(dict(Region=nombre, beta=float(m_r.params['IIF_lag1']),
                      se=float(m_r.std_errors['IIF_lag1']),
                      p=float(m_r.pvalues['IIF_lag1']),
                      N=int(m_r.nobs), N_dep=n_dep))
    total += int(m_r.nobs)
P('')
P(f'Suma de observaciones regionales : {total}')
P(f'N del modelo principal (B2)      : {RES["B2"]["N"]}')
P(f'N del modelo de referencia (A3)  : {RES["A3"]["N"]}')
if filas:
    P('')
    P(pd.DataFrame(filas).round(4).to_string(index=False))

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# SECCIÓN 12 — Descriptivas (Tabla 1 y Tabla 2)
# ═══════════════════════════════════════════════════════════════════════════
H('SECCIÓN 12 — Descriptivas')

VARS_DESC = ['crec_pib', 'crec_pib_pc', 'log_fintech_pc', 'd_acc_corresp',
             'd_acc_cta_ah', 'd_acc_internet', 'd_uso_pagos_pc', 'd_uso_transf_pc',
             'd_uso_depositos_pc', 'd_pro_micro', 'd_pro_cred_cons',
             'd_pro_cred_viv', 'educacion', 'densidad', 'informalidad',
             'IIF', 'log_IIF']
desc = panel[VARS_DESC].describe(percentiles=[.25, .5, .75]).T
desc['N'] = panel[VARS_DESC].notna().sum()
P('Tabla 1 — descriptivas del panel')
P(desc[['N', 'mean', 'std', 'min', '25%', '50%', '75%', 'max']].round(4).to_string())

P('')
P('Tabla 2 — evolución anual')
t2 = panel.groupby('anio').agg(
    N=('IIF', 'size'), IIF_prom=('IIF', 'mean'),
    crec_pib=('crec_pib', 'mean'), crec_pib_pc=('crec_pib_pc', 'mean'),
    log_IIF=('log_IIF', 'mean')).round(4)
P(t2.to_string())

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# SECCIÓN 13 — VERIFICACIÓN CONTRA EL DOCUMENTO
# ═══════════════════════════════════════════════════════════════════════════
H('SECCIÓN 13 — Verificación contra el documento')

vdf = pd.DataFrame(VERIF)
def _f(x):
    return '' if x is None or (isinstance(x, float) and x != x) else f'{x:,.4f}'
vdf['obtenido']  = vdf['obtenido'].map(_f)
vdf['documento'] = vdf['documento'].map(_f)
P(vdf[['item', 'obtenido', 'documento', 'estado', 'nota']].to_string(index=False))

P('')
disc = vdf[vdf['estado'] == 'DISCREPA']
P(f'TOTAL DISCREPANCIAS: {len(disc)}')
if len(disc):
    P('')
    P(disc[['item', 'obtenido', 'documento', 'nota']].to_string(index=False))

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# SECCIÓN 14 — Exportar
# ═══════════════════════════════════════════════════════════════════════════
H('SECCIÓN 14 — Exportación')
P(f'Filas de verificación: {len(vdf)}')
save_md(OUT_MD)